# Multi-station NO2 — the traffic diurnal cycle

Fetch **hourly-rollup NO2** for several stations in one metro area, then group by hour-of-day split into weekday vs. weekend to reveal the traffic-driven diurnal pattern. Shows multi-station handling and using the `lat`/`lon` columns to tell stations apart.

> Needs a free `OPENAQ_API_KEY`. The live cell is guarded so the notebook stays `--nbval-lax`-safe. Hourly rollups over a month are a few hundred rows per sensor — keep the window modest.

In [ ]:
import os

from earthlens import EarthLens

In [ ]:
bbox_lat = [34.0, 34.3]  # Los Angeles basin
bbox_lon = [-118.5, -118.1]
start, end = "2024-03-01", "2024-03-31"

In [ ]:
df = None
if os.environ.get("OPENAQ_API_KEY"):
    df = EarthLens(
        data_source="openaq",
        variables=["no2"],
        start=start,
        end=end,
        aoi=[bbox_lon[0], bbox_lat[0], bbox_lon[1], bbox_lat[1]],
        temporal_resolution="hourly",  # server-side hourly rollup
        max_locations=8,
        path="out/openaq",
    ).download(progress_bar=False)
    print(df.shape, "stations:", df['station_id'].nunique() if df is not None else 0)
else:
    print("set OPENAQ_API_KEY to run the live cell")

In [ ]:
diurnal = None
if df is not None and not df.empty:
    local = df.copy()
    local["hour"] = local["datetime_utc"].dt.hour
    local["is_weekend"] = local["datetime_utc"].dt.dayofweek >= 5
    diurnal = (
        local.groupby(["is_weekend", "hour"])["value"].mean().unstack("is_weekend")
    )
    diurnal.columns = ["weekday" if not c else "weekend" for c in diurnal.columns]
    display(diurnal.head())

In [ ]:
if diurnal is not None and not diurnal.empty:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 4))
    diurnal.plot(ax=ax, marker="o")
    ax.set_xlabel("hour of day (UTC)")
    ax.set_ylabel("NO2 (µg/m³)")
    ax.set_title("NO2 diurnal cycle — weekday vs weekend (metro mean)")
    ax.set_xticks(range(0, 24, 3))
    plt.show()